In [1]:
import numpy as np
import pandas as pd
from tensorflow.keras.datasets import mnist
from sklearn.cluster import KMeans
import pickle
def makefile(what,filename):
    with open(filename,"wb") as f3:
        pickle.dump(what,f3)

def readfile(filename):
    with open(filename,"rb") as f4:
        ans=pickle.load(f4)
    return ans
from math import *

import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

2026-01-06 10:59:04.665739: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-06 10:59:04.697628: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-06 10:59:05.536880: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/toshihayashi/anaconda3/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
from tensorflow.keras.datasets import fashion_mnist as fmnist

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

In [6]:
size=4

In [28]:
from tensorflow.keras.datasets import cifar10

In [7]:
import time

In [16]:
X_train=mnist.load_data()[0][0]
y_train=mnist.load_data()[0][1]
X_test=mnist.load_data()[1][0]
y_test=mnist.load_data()[1][1]
num_cluster=[2,5,10,25]
window_size=[15]
for size in window_size:
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for K in num_cluster:
        print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        #result=[]
        for c in range(1):
            print(c)
            start=time.time()
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            
            
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            end=time.time()
            tim=end-start
            print(tim)
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

50
0
86.92315912246704
50
0
86.74503803253174
50
0
192.43770265579224
50
0
244.85255312919617
50
0
344.2364037036896
50
0
405.89378118515015


In [ ]:
class Kmeans:
    def 

In [18]:
X_train=mnist.load_data()[0][0]
y_train=mnist.load_data()[0][1]
X_test=mnist.load_data()[1][0]
y_test=mnist.load_data()[1][1]
num_cluster=[2,5,10,25]
window_size=[15]
for size in window_size:
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for K in num_cluster:
        print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        #result=[]
        for c in range(1):
            #print(c)
            print("training stage")
            start=time.time()
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            GMM.fit(LCB)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            IF.fit(LCB)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            LOF.fit(LCB)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            OCSVM.fit(LCB)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            LP=LCB.mean(axis=0)
            end=time.time()
            tim=end-start
            print(tim)
            print("testing stage")
            start=time.time()
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            GMM_seen=GMM.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            IF_seen=IF.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            LOF_seen=LOF.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            print(tim)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

2
training stage
61.70798206329346
0.04048299789428711
0.11566758155822754
0.020569324493408203
0.7611510753631592
0.0001633167266845703
testing stage
4.783582925796509
0.0002644062042236328
0.00044345855712890625
0.12787747383117676
0.018573760986328125
0.8550636768341064
5
training stage
109.72584342956543
0.0526273250579834
0.11756443977355957
0.07184672355651855
0.6390855312347412
0.00020194053649902344
testing stage
5.002343654632568
0.0003190040588378906
0.00064849853515625
0.13695597648620605
0.07669210433959961
0.8828442096710205
10
training stage
121.72013807296753
0.09248542785644531
0.11675858497619629
0.19897723197937012
0.6729109287261963
0.00020885467529296875
testing stage
4.86253023147583
0.00031685829162597656
0.0007274150848388672
0.14045119285583496
0.2688724994659424
1.2566041946411133
25
training stage
292.85369634628296
0.007985115051269531
0.12015008926391602
0.04852294921875
0.8060708045959473
0.0001983642578125
testing stage
5.134796380996704
0.0004534721374511

In [13]:
X_train=mnist.load_data()[0][0]
y_train=mnist.load_data()[0][1]
X_test=mnist.load_data()[1][0]
y_test=mnist.load_data()[1][1]
num_cluster=[100]
window_size=[2,3,5,7,11,15]
for K in num_cluster:
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for size in window_size:
        print(size)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        #result=[]
        for c in range(10):
            #print(c)
            print("training stage")
            start=time.time()
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            GMM.fit(LCB)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            IF.fit(LCB)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            LOF.fit(LCB)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            OCSVM.fit(LCB)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            LP=LCB.mean(axis=0)
            end=time.time()
            tim=end-start
            #print(tim)
            print("testing stage")
            start=time.time()
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            end=time.time()
            tim=end-start
            #rint(tim)
            start=time.time()
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            GMM_seen=GMM.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            IF_seen=IF.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            LOF_seen=LOF.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            #print(tim)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

2
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
3
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
5
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
7
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
traini

In [14]:
Classic_table

[[69.5, 96.8, 63.7, 68.8, 67.6, 70.7, 63.7, 73.7, 68.0, 71.8],
 [74.1, 96.4, 69.9, 76.9, 71.4, 77.0, 68.3, 75.3, 72.4, 75.0],
 [78.1, 94.5, 76.9, 83.8, 76.6, 81.2, 76.6, 74.9, 81.4, 81.3],
 [79.9, 93.8, 80.9, 86.7, 79.3, 87.1, 82.0, 75.4, 86.1, 84.4],
 [91.0, 70.5, 86.3, 90.6, 73.0, 93.9, 84.7, 75.0, 86.9, 84.0],
 [89.6, 41.8, 72.1, 86.7, 69.4, 75.4, 80.6, 81.6, 83.6, 77.4]]

In [15]:
GMM_table

[[56.5, 98.9, 54.8, 73.2, 69.9, 71.2, 61.5, 79.2, 56.0, 67.4],
 [65.4, 99.3, 64.6, 85.4, 79.1, 80.7, 69.9, 83.7, 67.1, 75.9],
 [86.1, 99.6, 86.1, 97.0, 92.9, 93.4, 91.4, 93.0, 89.2, 92.3],
 [95.8, 99.6, 92.2, 98.2, 95.7, 97.4, 97.2, 96.5, 95.5, 96.3],
 [99.2, 99.6, 97.3, 97.5, 96.0, 98.2, 98.7, 97.4, 97.1, 97.8],
 [99.3, 99.2, 96.8, 96.9, 95.1, 97.4, 98.8, 98.1, 96.2, 96.7]]

In [20]:
np.array(GMM_table).mean(axis=1)

array([68.86, 77.11, 92.1 , 96.44, 97.88, 97.45])

In [16]:
IF_table

[[55.0, 99.4, 62.3, 79.7, 68.9, 80.4, 66.6, 77.4, 58.6, 70.0],
 [60.7, 99.4, 74.5, 86.0, 75.2, 86.9, 73.2, 83.8, 67.2, 76.0],
 [71.0, 99.4, 83.8, 93.4, 90.3, 87.8, 83.2, 86.4, 80.5, 88.1],
 [81.9, 99.3, 87.0, 93.1, 93.5, 94.7, 88.7, 90.6, 90.6, 92.9],
 [96.2, 99.1, 89.8, 91.8, 90.9, 96.0, 91.8, 95.3, 90.9, 94.2],
 [92.5, 97.6, 83.1, 90.4, 86.6, 90.2, 87.4, 94.9, 77.0, 91.1]]

In [18]:
LOF_table

[[74.0, 80.5, 65.2, 80.5, 75.7, 78.7, 70.5, 76.9, 77.1, 70.3],
 [86.1, 78.1, 73.9, 90.3, 87.0, 87.0, 78.2, 79.9, 85.8, 80.7],
 [92.4, 86.2, 88.7, 97.4, 96.5, 93.3, 90.4, 90.0, 95.1, 91.0],
 [96.6, 84.1, 94.1, 98.5, 98.2, 97.6, 97.0, 93.2, 97.0, 96.4],
 [94.2, 96.4, 97.9, 98.4, 96.7, 98.3, 99.2, 96.7, 96.3, 98.1],
 [95.0, 99.5, 94.9, 96.2, 95.0, 95.6, 96.4, 97.7, 89.2, 96.8]]

In [19]:
OCSVM_table

[[71.4, 96.2, 61.3, 60.0, 66.2, 60.5, 58.7, 71.0, 65.6, 68.4],
 [76.5, 96.9, 64.3, 64.2, 67.4, 64.1, 60.8, 72.6, 68.9, 70.4],
 [83.1, 96.3, 69.2, 70.5, 70.4, 66.4, 66.6, 74.7, 73.3, 74.4],
 [87.3, 97.0, 74.2, 76.4, 72.5, 72.5, 70.4, 73.9, 76.9, 74.7],
 [94.6, 93.6, 88.8, 92.9, 83.8, 93.3, 91.2, 82.9, 88.5, 90.5],
 [95.1, 74.4, 85.2, 92.1, 85.6, 87.7, 92.9, 92.4, 88.0, 90.2]]

In [21]:
X_train=fmnist.load_data()[0][0]
y_train=fmnist.load_data()[0][1]
X_test=fmnist.load_data()[1][0]
y_test=fmnist.load_data()[1][1]
num_cluster=[100]
window_size=[2,3,5,7,11,15]
for K in num_cluster:
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for size in window_size:
        print(size)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        #result=[]
        for c in range(10):
            #print(c)
            print("training stage")
            start=time.time()
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            GMM.fit(LCB)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            IF.fit(LCB)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            LOF.fit(LCB)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            OCSVM.fit(LCB)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            LP=LCB.mean(axis=0)
            end=time.time()
            tim=end-start
            #print(tim)
            print("testing stage")
            start=time.time()
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            end=time.time()
            tim=end-start
            #rint(tim)
            start=time.time()
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            GMM_seen=GMM.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            IF_seen=IF.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            LOF_seen=LOF.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            #print(tim)
            start=time.time()
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            #print(tim)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

2
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
3
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
5
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
7
training stage
testing stage
training stage
testing stage
training stage
testing stage
training stage
testing stage
traini

In [22]:
Classic_table

[[51.1, 92.7, 55.1, 72.4, 68.7, 91.6, 50.3, 95.2, 63.4, 86.9],
 [57.6, 93.4, 55.8, 77.0, 70.5, 92.4, 50.8, 95.5, 65.7, 88.9],
 [67.1, 93.4, 61.3, 82.8, 73.9, 94.1, 52.2, 95.5, 70.6, 93.2],
 [68.7, 93.5, 64.0, 82.6, 74.9, 95.3, 53.3, 95.8, 73.4, 95.4],
 [73.7, 94.9, 65.1, 75.7, 76.6, 96.3, 53.2, 96.6, 78.6, 96.6],
 [79.2, 96.6, 70.4, 77.8, 81.3, 96.2, 61.9, 96.7, 83.0, 96.8]]

In [23]:
GMM_table

[[72.4, 96.3, 68.7, 84.0, 79.3, 93.3, 64.7, 96.4, 59.5, 92.3],
 [81.3, 96.6, 74.9, 88.4, 83.1, 93.4, 72.4, 97.2, 72.1, 94.0],
 [84.4, 95.2, 78.8, 93.4, 84.7, 93.7, 77.2, 97.6, 79.8, 95.2],
 [85.7, 95.4, 80.7, 94.4, 85.9, 93.2, 79.9, 97.8, 86.5, 97.3],
 [84.7, 93.9, 81.4, 91.2, 85.0, 92.8, 80.0, 97.8, 94.5, 98.5],
 [81.9, 91.8, 81.8, 89.0, 84.2, 93.0, 77.1, 96.5, 93.9, 98.6]]

In [24]:
LOF_table

[[70.6, 90.5, 79.8, 85.5, 80.6, 95.2, 72.5, 94.2, 60.0, 92.5],
 [71.0, 90.8, 79.2, 89.4, 80.7, 96.7, 74.0, 94.9, 63.1, 93.6],
 [78.5, 89.6, 80.6, 93.1, 85.4, 95.8, 77.6, 94.9, 76.9, 96.4],
 [80.9, 87.0, 81.5, 92.1, 85.6, 93.8, 79.2, 95.5, 88.5, 96.2],
 [81.9, 83.9, 83.8, 90.9, 83.4, 90.3, 77.9, 95.0, 92.0, 97.3],
 [77.1, 56.8, 84.1, 89.5, 80.6, 87.4, 77.5, 94.6, 86.3, 96.7]]

In [25]:
OCSVM_table

[[54.6, 89.3, 69.9, 67.5, 73.9, 86.4, 63.4, 92.9, 68.0, 85.0],
 [54.4, 92.9, 71.4, 68.0, 71.6, 87.4, 60.0, 93.4, 68.0, 85.2],
 [60.9, 95.2, 76.2, 73.3, 71.5, 90.2, 59.5, 94.2, 70.2, 90.2],
 [68.7, 96.0, 71.6, 80.7, 71.9, 93.2, 59.3, 94.8, 71.4, 93.9],
 [76.9, 97.6, 69.0, 85.0, 74.5, 94.6, 58.0, 97.0, 76.9, 97.4],
 [82.9, 97.6, 79.0, 83.3, 84.0, 94.5, 66.6, 97.3, 83.0, 98.1]]

In [26]:
IF_table

[[68.7, 95.3, 59.9, 76.2, 69.4, 87.0, 60.8, 91.9, 54.3, 84.7],
 [76.6, 92.0, 63.4, 81.7, 79.7, 83.6, 67.8, 95.3, 59.7, 86.9],
 [80.3, 74.8, 67.0, 88.0, 80.0, 87.2, 71.7, 95.5, 77.2, 92.6],
 [75.8, 62.5, 70.9, 81.4, 77.6, 88.0, 72.2, 94.8, 78.9, 92.4],
 [63.2, 43.7, 56.2, 48.0, 62.3, 62.3, 62.6, 86.2, 77.4, 93.8],
 [46.5, 12.9, 47.4, 40.4, 47.6, 22.5, 53.3, 42.8, 53.0, 61.1]]

In [29]:
X_train=cifar10.load_data()[0][0]
y_train=cifar10.load_data()[0][1]
X_test=cifar10.load_data()[1][0]
y_test=cifar10.load_data()[1][1]
y_train=y_train.reshape(len(y_train))
y_test=y_test.reshape(len(y_test))
num_cluster=[100]
window_size=[2,3,5,7,11]
for K in num_cluster:
    print(K)
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for size in window_size:
        print(size)
        
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        for c in range(10):
            print(c)
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num*3:num*3+3]=train[:,i:32-size+1+i,j:32-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num*3:num*3+3]=X_test[:,i:32-size+1+i,j:32-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
        print(Classic_result)
        print(GMM_result)
        print(IF_result)
        print(LOF_result)
        print(OCSVM_result)

100
2
0
1
2
3
4
5
6
7
8
9
[40.0, 77.9, 40.3, 60.8, 48.8, 59.7, 60.1, 65.1, 66.5, 79.4]
[52.0, 73.5, 48.0, 65.4, 58.2, 66.9, 67.4, 59.0, 71.6, 74.6]
[68.1, 68.8, 65.8, 61.0, 63.0, 63.5, 59.2, 53.1, 71.6, 64.5]
[45.5, 75.4, 52.9, 61.3, 55.3, 59.9, 68.4, 61.2, 64.4, 74.4]
[32.7, 75.0, 36.0, 60.5, 47.9, 58.0, 60.0, 61.8, 57.9, 77.2]
3
0
1
2
3
4
5
6
7
8
9
[40.4, 77.1, 40.1, 60.6, 48.4, 60.4, 59.6, 65.8, 65.7, 79.3]
[55.2, 74.8, 49.6, 66.0, 61.4, 66.5, 71.0, 61.9, 73.0, 77.1]
[67.0, 63.5, 65.1, 59.5, 71.0, 60.5, 64.3, 51.5, 69.5, 62.5]
[45.1, 77.1, 53.5, 61.4, 54.1, 61.5, 67.1, 63.0, 64.1, 76.1]
[33.2, 75.4, 35.7, 60.2, 46.6, 58.7, 58.7, 63.7, 57.0, 77.6]
5
0
1
2
3
4
5
6
7
8
9
[42.3, 76.1, 40.1, 59.9, 48.7, 62.0, 59.0, 66.3, 63.2, 77.7]
[55.7, 75.5, 52.1, 65.0, 64.5, 65.7, 72.6, 63.4, 72.2, 77.3]
[56.4, 53.4, 61.3, 53.3, 71.1, 46.8, 63.5, 45.9, 65.9, 56.3]
[50.8, 77.7, 53.9, 61.8, 52.7, 65.8, 64.8, 65.0, 63.6, 75.5]
[37.3, 76.5, 36.2, 59.9, 44.1, 61.5, 56.6, 65.4, 56.4, 76.3]
7
0
1
2
3
4
5
6

In [ ]:
X_train=cifar10.load_data()[0][0]
y_train=cifar10.load_data()[0][1]
X_test=cifar10.load_data()[1][0]
y_test=cifar10.load_data()[1][1]
y_train=y_train.reshape(len(y_train))
y_test=y_test.reshape(len(y_test))
num_cluster=[50]
window_size=[2,3,5,7,11]
for K in num_cluster:
    print(K)
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for c in range(10):
        print(c)
        train = X_train[y_train==c]
        position=np.array(range(size**2)).reshape(size,size)
        windows=np.zeros([len(train),32-size+1,32-size+1,size*size*3])
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        full_LCB=[]
        for size in window_size:
            print(size)
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num*3:num*3+3]=train[:,i:32-size+1+i,j:32-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            
            Classic_result=[]
            GMM_result=[]
            IF_result=[]
            LOF_result=[]
            OCSVM_result=[]
        
            #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num*3:num*3+3]=X_test[:,i:32-size+1+i,j:32-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
        print(Classic_result)
        print(GMM_result)
        print(IF_result)
        print(LOF_result)
        print(OCSVM_result)

In [30]:
Classic_table

[[40.0, 77.9, 40.3, 60.8, 48.8, 59.7, 60.1, 65.1, 66.5, 79.4],
 [40.4, 77.1, 40.1, 60.6, 48.4, 60.4, 59.6, 65.8, 65.7, 79.3],
 [42.3, 76.1, 40.1, 59.9, 48.7, 62.0, 59.0, 66.3, 63.2, 77.7],
 [44.6, 74.7, 40.8, 58.9, 48.5, 63.1, 58.1, 66.6, 61.5, 75.3],
 [49.5, 69.9, 45.8, 58.7, 50.3, 63.0, 57.5, 65.8, 60.0, 71.1]]

In [31]:
GMM_table

[[52.0, 73.5, 48.0, 65.4, 58.2, 66.9, 67.4, 59.0, 71.6, 74.6],
 [55.2, 74.8, 49.6, 66.0, 61.4, 66.5, 71.0, 61.9, 73.0, 77.1],
 [55.7, 75.5, 52.1, 65.0, 64.5, 65.7, 72.6, 63.4, 72.2, 77.3],
 [57.6, 74.9, 54.0, 62.3, 65.7, 66.2, 74.3, 63.9, 70.9, 75.4],
 [60.1, 72.6, 58.9, 59.0, 65.7, 62.5, 70.7, 65.4, 68.9, 71.7]]

In [32]:
LOF_table

[[45.5, 75.4, 52.9, 61.3, 55.3, 59.9, 68.4, 61.2, 64.4, 74.4],
 [45.1, 77.1, 53.5, 61.4, 54.1, 61.5, 67.1, 63.0, 64.1, 76.1],
 [50.8, 77.7, 53.9, 61.8, 52.7, 65.8, 64.8, 65.0, 63.6, 75.5],
 [51.5, 77.4, 53.5, 61.4, 51.2, 68.5, 65.9, 65.9, 61.6, 74.4],
 [55.6, 74.8, 57.4, 60.3, 52.5, 68.2, 64.1, 68.5, 61.4, 73.9]]

In [33]:
OCSVM_table

[[32.7, 75.0, 36.0, 60.5, 47.9, 58.0, 60.0, 61.8, 57.9, 77.2],
 [33.2, 75.4, 35.7, 60.2, 46.6, 58.7, 58.7, 63.7, 57.0, 77.6],
 [37.3, 76.5, 36.2, 59.9, 44.1, 61.5, 56.6, 65.4, 56.4, 76.3],
 [41.0, 75.9, 37.6, 59.7, 43.3, 63.8, 55.2, 66.1, 57.5, 74.4],
 [49.0, 71.9, 43.7, 58.8, 44.5, 65.2, 52.4, 66.8, 59.2, 69.8]]

In [34]:
IF_table

[[68.1, 68.8, 65.8, 61.0, 63.0, 63.5, 59.2, 53.1, 71.6, 64.5],
 [67.0, 63.5, 65.1, 59.5, 71.0, 60.5, 64.3, 51.5, 69.5, 62.5],
 [56.4, 53.4, 61.3, 53.3, 71.1, 46.8, 63.5, 45.9, 65.9, 56.3],
 [55.9, 46.7, 60.9, 52.4, 69.7, 43.9, 68.2, 46.1, 62.2, 49.8],
 [52.7, 44.9, 57.4, 48.3, 64.2, 41.6, 68.1, 42.3, 49.8, 45.4]]

In [35]:
X_train=cifar10.load_data()[0][0]
y_train=cifar10.load_data()[0][1]
X_test=cifar10.load_data()[1][0]
y_test=cifar10.load_data()[1][1]
y_train=y_train.reshape(len(y_train))
y_test=y_test.reshape(len(y_test))
num_cluster=[100]
window_size=[15]
for K in num_cluster:
    print(K)
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for size in window_size:
        print(size)
        
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        for c in range(10):
            print(c)
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num*3:num*3+3]=train[:,i:32-size+1+i,j:32-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num*3:num*3+3]=X_test[:,i:32-size+1+i,j:32-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
        print(Classic_result)
        print(GMM_result)
        print(IF_result)
        print(LOF_result)
        print(OCSVM_result)

100
15
0
1
2
3
4
5
6
7
8
9
[51.8, 66.1, 47.4, 58.4, 51.3, 61.6, 55.6, 66.5, 60.5, 70.2]
[58.6, 71.3, 58.6, 57.9, 64.6, 58.9, 68.3, 65.5, 68.1, 70.6]
[44.7, 43.6, 57.4, 45.7, 62.6, 40.2, 58.0, 41.7, 43.9, 43.7]
[57.3, 72.6, 59.1, 60.0, 53.7, 66.0, 64.8, 66.6, 63.8, 74.0]
[54.6, 68.1, 46.8, 58.1, 45.4, 64.1, 50.5, 67.1, 60.7, 68.5]


In [12]:
Classic_result

[]

In [10]:
Classic_table

[[89.2]]

In [20]:
X_train=cifar10.load_data()[0][0]
y_train=cifar10.load_data()[0][1]
X_test=cifar10.load_data()[1][0]
y_test=cifar10.load_data()[1][1]
y_train=y_train.reshape(len(y_train))
y_test=y_test.reshape(len(y_test))
num_cluster=[50]
window_size=[2,3,5,7,11,15]
for size in window_size:
    print(size)
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]

    for K in num_cluster:
        #print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        for c in range(1):
            #print(c)
            print("training stage")
            start=time.time()
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num*3:num*3+3]=train[:,i:32-size+1+i,j:32-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            GMM.fit(LCB)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            IF.fit(LCB)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            LOF.fit(LCB)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            OCSVM.fit(LCB)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            LP=LCB.mean(axis=0)
            end=time.time()
            tim=end-start
            print(tim)
            print("testing stage")
            start=time.time()
            windows_test=np.zeros([len(X_test),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num*3:num*3+3]=X_test[:,i:32-size+1+i,j:32-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            GMM_seen=GMM.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            IF_seen=IF.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            LOF_seen=LOF.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            print(tim)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
    #makefile(Classic_table,"result/CIFAR10_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/CIFAR10_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/CIFAR10_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/CIFAR10_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/CIFAR10_OCLCB-OCSVM-size"+str(size)+".pkl")

2
training stage
138.13664841651917
0.008486747741699219
0.11402344703674316
0.0325167179107666
0.6046397686004639
0.0002143383026123047
testing stage
2.788024425506592
0.0007426738739013672
0.0017156600952148438
0.15544986724853516
0.030473947525024414
1.1342496871948242
3
training stage
218.58490467071533
0.00849461555480957
0.11403417587280273
0.035124778747558594
0.6059143543243408
0.0001652240753173828
testing stage
3.3962111473083496
0.0008368492126464844
0.0019974708557128906
0.18210864067077637
0.03858017921447754
1.1421828269958496
5
training stage
450.30088543891907
0.008460760116577148
0.11453819274902344
0.032750606536865234
0.6065871715545654
0.0001652240753173828
testing stage
4.950579404830933
0.0007488727569580078
0.0016362667083740234
0.15842771530151367
0.030257463455200195
1.1218416690826416
7
training stage
719.5981709957123
0.008978605270385742
0.11400222778320312
0.03252553939819336
0.6080648899078369
0.0001666545867919922
testing stage
7.118736267089844
0.0007379

In [21]:
X_train=cifar10.load_data()[0][0]
y_train=cifar10.load_data()[0][1]
X_test=cifar10.load_data()[1][0]
y_test=cifar10.load_data()[1][1]
y_train=y_train.reshape(len(y_train))
y_test=y_test.reshape(len(y_test))
num_cluster=[2,5,10,25]
window_size=[15]
for size in window_size:
    print(size)
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]

    for K in num_cluster:
        #print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        for c in range(1):
            #print(c)
            print("training stage")
            start=time.time()
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num*3:num*3+3]=train[:,i:32-size+1+i,j:32-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            GMM.fit(LCB)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            IF.fit(LCB)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            LOF.fit(LCB)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            OCSVM.fit(LCB)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            LP=LCB.mean(axis=0)
            end=time.time()
            tim=end-start
            print(tim)
            print("testing stage")
            start=time.time()
            windows_test=np.zeros([len(X_test),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num*3:num*3+3]=X_test[:,i:32-size+1+i,j:32-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            GMM_seen=GMM.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            IF_seen=IF.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            LOF_seen=LOF.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            print(tim)
            start=time.time()
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            end=time.time()
            tim=end-start
            print(tim)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
    #makefile(Classic_table,"result/CIFAR10_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/CIFAR10_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/CIFAR10_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/CIFAR10_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/CIFAR10_OCLCB-OCSVM-size"+str(size)+".pkl")

15
training stage
93.4419116973877
0.00478363037109375
0.11527228355407715
0.02141118049621582
0.4787893295288086
0.00015783309936523438
testing stage
14.346150636672974
0.000274658203125
0.0004379749298095703
0.12897038459777832
0.026669740676879883
0.7472984790802002
training stage
330.8503477573395
0.005099773406982422
0.11601901054382324
0.03945279121398926
0.4760410785675049
0.00024247169494628906
testing stage
14.579610347747803
0.00027298927307128906
0.0006098747253417969
0.14165067672729492
0.05044198036193848
0.746955394744873
training stage
596.6763429641724
0.0056726932525634766
0.1150062084197998
0.12587761878967285
0.47258973121643066
0.000164031982421875
testing stage
14.460643529891968
0.0003440380096435547
0.0007317066192626953
0.13555049896240234
0.18876934051513672
1.0391972064971924
training stage
1462.9164350032806
0.007229328155517578
0.11388778686523438
0.031060457229614258
0.5776102542877197
0.0001747608184814453
testing stage
14.590282917022705
0.000451326370239

In [ ]:
X_train=mnist.load_data()[0][0]
y_train=mnist.load_data()[0][1]
X_test=mnist.load_data()[1][0]
y_test=mnist.load_data()[1][1]
num_cluster=[2,3,5,8,10,16]
window_size=[5,7,11,15]
for size in window_size:
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for K in num_cluster:
        print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        #result=[]
        for c in range(1):
            print(c)
            start=time.time()
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            end=time.time()
            
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
    makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

In [5]:
X_train=mnist.load_data()[0][0]
y_train=mnist.load_data()[0][1]
X_test=mnist.load_data()[1][0]
y_test=mnist.load_data()[1][1]
num_cluster=[50]
window_size=[2,3,5,7,11,15]
for size in window_size:
    print(size)
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for K in num_cluster:
        #print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        #result=[]
        for c in range(10):
            print(c)
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
        print(Classic_result)
        print(GMM_result)
        print(IF_result)
        print(LOF_result)
        print(OCSVM_result)
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

5
0
1
2
3
4
5
6
7
8
9
[81.4, 93.4, 76.4, 80.6, 73.9, 80.8, 74.2, 79.1, 78.4, 78.5]
[86.6, 99.3, 83.6, 92.5, 92.2, 91.6, 88.1, 92.3, 84.6, 91.2]
[79.0, 99.4, 81.9, 89.6, 86.1, 89.8, 82.2, 87.3, 81.9, 86.1]
[91.6, 91.4, 87.0, 94.0, 95.4, 92.3, 88.7, 90.4, 89.4, 89.9]
[83.0, 95.9, 70.6, 71.7, 71.9, 70.1, 67.2, 76.5, 73.9, 74.9]
7
0
1
2
3
4
5
6
7
8
9
[86.1, 90.6, 79.7, 83.1, 72.5, 85.4, 79.5, 80.2, 81.0, 82.5]
[93.1, 99.4, 89.2, 95.3, 93.6, 93.8, 95.7, 96.1, 90.0, 95.9]
[85.1, 99.1, 85.0, 91.8, 87.0, 92.6, 88.1, 89.3, 86.4, 90.9]
[93.0, 90.9, 91.8, 96.5, 95.3, 94.9, 96.5, 93.0, 92.7, 95.7]
[88.6, 95.5, 75.4, 76.4, 70.5, 76.9, 72.4, 77.8, 76.6, 77.1]
11
0
1
2
3
4
5
6
7
8
9
[92.1, 70.4, 84.0, 86.4, 77.1, 91.7, 88.5, 82.1, 80.9, 81.4]
[98.5, 99.4, 94.1, 95.4, 93.6, 97.1, 97.4, 96.1, 92.1, 96.4]
[95.2, 99.2, 89.7, 91.5, 89.7, 96.5, 93.2, 90.5, 85.9, 92.3]
[93.2, 98.5, 95.9, 96.9, 93.8, 97.1, 98.1, 96.0, 89.3, 97.5]
[94.7, 96.0, 85.4, 88.9, 84.6, 93.1, 93.0, 87.9, 83.3, 90.3]
15
0
1
2
3
4
5
6
7

In [6]:
X_train=mnist.load_data()[0][0]
y_train=mnist.load_data()[0][1]
X_test=mnist.load_data()[1][0]
y_test=mnist.load_data()[1][1]
num_cluster=[50]
window_size=[5,7,11,15]
for size in window_size:
    print(size)
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for K in num_cluster:
        #print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        #result=[]
        for c in range(10):
            #print(c)
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
        print(Classic_result)
        print(GMM_result)
        print(IF_result)
        print(LOF_result)
        print(OCSVM_result)
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

5
[79.6, 93.3, 75.7, 82.1, 74.3, 80.4, 76.2, 74.8, 79.8, 79.3]
[88.2, 99.4, 85.4, 96.1, 93.5, 93.1, 91.8, 93.5, 88.5, 93.0]
[77.7, 99.4, 80.5, 92.7, 90.0, 91.1, 83.6, 88.5, 82.2, 89.0]
[93.2, 86.4, 87.6, 96.7, 96.7, 92.8, 91.2, 90.4, 93.1, 91.8]
[83.3, 96.0, 69.7, 71.1, 71.1, 68.1, 67.1, 75.1, 73.5, 74.7]
7
[82.2, 90.7, 81.0, 86.4, 75.1, 86.8, 82.7, 76.3, 84.4, 82.9]
[96.2, 99.5, 92.5, 97.4, 94.8, 97.0, 97.4, 96.4, 94.5, 96.5]
[83.2, 99.3, 87.0, 92.6, 90.4, 94.9, 91.4, 91.0, 88.9, 93.4]
[96.3, 88.0, 93.6, 98.2, 97.2, 97.2, 97.0, 93.3, 96.2, 96.5]
[87.3, 96.4, 75.5, 77.4, 71.7, 75.2, 72.7, 74.8, 77.3, 75.8]
11
[93.8, 73.0, 86.2, 89.4, 78.4, 94.4, 89.7, 77.4, 85.4, 83.0]
[98.7, 99.4, 95.8, 96.9, 95.1, 98.1, 98.5, 97.0, 95.0, 97.2]
[96.5, 98.7, 90.9, 93.0, 90.5, 97.2, 94.6, 94.0, 90.2, 95.0]
[92.4, 98.1, 95.2, 97.9, 95.8, 97.7, 99.0, 96.2, 93.0, 97.8]
[95.2, 92.5, 87.6, 91.5, 85.2, 94.6, 93.6, 87.6, 85.8, 92.1]
15
[91.3, 37.4, 80.1, 89.1, 74.7, 82.6, 83.7, 85.4, 86.7, 80.7]
[99.0, 98.9, 9

In [5]:
X_train=mnist.load_data()[0][0]
y_train=mnist.load_data()[0][1]
X_test=mnist.load_data()[1][0]
y_test=mnist.load_data()[1][1]
num_cluster=[50]
window_size=[2,3,4]
for size in window_size:
    print(size)
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for K in num_cluster:
        #print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        #result=[]
        for c in range(10):
            #print(c)
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
        print(Classic_result)
        print(GMM_result)
        print(IF_result)
        print(LOF_result)
        print(OCSVM_result)
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

2
[71.1, 96.1, 65.0, 69.9, 67.5, 72.1, 64.6, 73.9, 69.0, 71.6]
[68.3, 98.7, 60.7, 78.4, 73.9, 76.1, 64.6, 80.1, 66.3, 71.5]
[63.8, 99.3, 70.3, 80.0, 73.0, 82.3, 72.2, 80.9, 68.5, 75.4]
[80.1, 83.0, 68.6, 83.9, 79.3, 83.0, 73.2, 79.9, 81.0, 73.8]
[71.3, 96.0, 61.3, 60.4, 66.0, 61.3, 58.8, 71.2, 65.5, 68.7]
3
[75.3, 96.0, 70.8, 78.0, 71.1, 78.7, 68.4, 75.8, 73.5, 74.7]
[71.3, 99.3, 68.0, 87.5, 81.2, 83.6, 71.5, 84.5, 74.0, 78.2]
[71.1, 99.3, 76.9, 89.9, 81.2, 85.5, 77.6, 84.1, 74.1, 79.8]
[88.0, 82.6, 77.4, 91.2, 88.2, 88.1, 79.6, 84.0, 86.6, 81.3]
[76.2, 96.8, 64.8, 65.4, 68.0, 65.4, 61.3, 73.1, 69.3, 70.7]
4
[78.8, 95.2, 73.6, 81.5, 73.4, 79.6, 72.3, 74.9, 76.7, 77.4]
[81.3, 99.4, 79.8, 94.0, 89.8, 89.0, 82.3, 90.1, 83.8, 88.0]
[74.3, 99.5, 78.7, 92.5, 87.0, 90.6, 79.8, 86.4, 78.6, 86.4]
[91.7, 86.3, 83.0, 95.2, 94.2, 90.9, 84.7, 87.3, 90.9, 88.5]
[80.5, 96.7, 67.4, 69.0, 69.6, 66.6, 64.2, 74.5, 71.8, 73.0]


In [6]:
X_train=fmnist.load_data()[0][0]
y_train=fmnist.load_data()[0][1]
X_test=fmnist.load_data()[1][0]
y_test=fmnist.load_data()[1][1]
num_cluster=[50]
window_size=[2,3,4]
for size in window_size:
    print(size)
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for K in num_cluster:
        print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        #result=[]
        for c in range(10):
            #print(c)
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
        print(Classic_result)
        print(GMM_result)
        print(IF_result)
        print(LOF_result)
        print(OCSVM_result)
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

2
50
[50.8, 91.9, 54.6, 71.4, 68.1, 91.7, 49.0, 94.8, 64.4, 86.6]
[74.8, 96.0, 70.6, 84.2, 81.9, 93.9, 66.4, 96.5, 63.3, 93.3]
[73.1, 94.6, 61.9, 81.4, 79.0, 91.1, 57.1, 94.1, 57.5, 91.8]
[72.5, 90.3, 79.8, 85.9, 80.0, 95.2, 73.4, 94.7, 58.7, 91.4]
[51.8, 89.8, 68.3, 66.4, 70.0, 87.5, 59.9, 92.8, 67.4, 84.0]
3
50
[56.4, 92.0, 56.2, 76.2, 69.7, 92.6, 49.7, 95.2, 66.5, 88.7]
[81.2, 95.8, 74.6, 89.4, 83.6, 94.3, 72.0, 97.2, 73.6, 94.5]
[76.5, 91.4, 67.5, 86.6, 81.4, 91.2, 69.9, 96.1, 67.3, 93.8]
[71.6, 90.4, 79.6, 89.1, 80.6, 96.7, 74.1, 95.1, 66.6, 93.5]
[53.6, 92.2, 70.5, 68.0, 68.7, 88.7, 57.5, 93.1, 68.2, 85.2]
4
50
[61.3, 91.6, 59.5, 79.4, 72.4, 93.4, 51.3, 95.3, 69.5, 91.2]
[82.8, 94.4, 77.0, 91.8, 83.7, 94.7, 75.6, 97.2, 78.6, 94.7]
[84.1, 86.1, 72.9, 86.6, 83.1, 92.8, 74.8, 96.7, 72.7, 94.7]
[76.0, 89.3, 82.8, 91.4, 82.8, 96.0, 76.9, 95.6, 77.0, 94.6]
[57.2, 94.2, 73.7, 70.3, 69.2, 89.8, 58.4, 93.9, 69.1, 87.2]


In [ ]:
LCB  k=50, d, s=1

LCB d 

In [7]:
X_train=fmnist.load_data()[0][0]
y_train=fmnist.load_data()[0][1]
X_test=fmnist.load_data()[1][0]
y_test=fmnist.load_data()[1][1]
num_cluster=[25,50]
window_size=[5,7,11,15]
for size in window_size:
    print(size)
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for K in num_cluster:
        print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        #result=[]
        for c in range(1):
            #print(c)
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
        print(Classic_result)
        print(GMM_result)
        print(IF_result)
        print(LOF_result)
        print(OCSVM_result)
    #makefile(Classic_table,"result/MNIST_OCLCB-Classic-size"+str(size)+".pkl")
    #makefile(GMM_table,"result/MNIST_OCLCB-GMM-size"+str(size)+".pkl")
    #makefile(IF_table,"result/MNIST_OCLCB-IF-size"+str(size)+".pkl")
    #makefile(LOF_table,"result/MNIST_OCLCB-LOF-size"+str(size)+".pkl")
    #makefile(OCSVM_table,"result/MNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

5
25
[64.4, 93.0, 58.6, 77.9, 72.6, 93.7, 49.4, 95.8, 72.8, 92.3]
[82.9, 95.5, 76.1, 93.4, 83.8, 93.8, 75.4, 97.5, 78.5, 94.7]
[82.0, 94.0, 74.2, 91.1, 83.5, 93.9, 75.8, 97.7, 84.6, 94.5]
[77.8, 83.8, 82.6, 90.3, 80.9, 93.5, 80.3, 94.7, 82.5, 94.3]
[65.2, 94.9, 76.5, 72.4, 71.4, 91.5, 60.7, 94.1, 70.7, 90.5]
50
[64.6, 92.6, 60.1, 80.2, 73.4, 93.8, 50.6, 95.6, 71.3, 93.2]
[82.9, 95.6, 78.0, 92.7, 84.2, 94.0, 77.5, 97.4, 80.0, 95.6]
[80.1, 92.0, 74.1, 90.5, 85.7, 92.0, 75.8, 97.0, 84.9, 95.3]
[78.8, 90.9, 84.1, 91.3, 83.0, 94.4, 78.2, 95.9, 82.8, 95.4]
[61.7, 95.2, 74.5, 72.4, 70.0, 90.8, 59.0, 94.4, 69.9, 90.2]
7
25
[63.5, 95.3, 53.1, 75.0, 71.2, 94.3, 46.2, 96.2, 75.2, 94.9]
[84.8, 95.4, 75.1, 92.6, 83.2, 93.3, 76.9, 97.4, 84.4, 96.6]
[85.2, 95.9, 77.9, 90.6, 85.3, 92.2, 79.4, 97.3, 92.3, 96.0]
[77.3, 87.9, 81.6, 88.8, 81.1, 92.6, 80.3, 94.7, 86.0, 95.0]
[71.5, 95.9, 69.9, 77.6, 74.3, 93.4, 61.4, 95.7, 74.1, 95.4]
50
[67.8, 93.3, 58.9, 76.3, 73.5, 94.9, 49.2, 96.2, 73.1, 94.4]
[85.8, 9

In [9]:
X_train=cifar10.load_data()[0][0]
y_train=cifar10.load_data()[0][1]
X_test=cifar10.load_data()[1][0]
y_test=cifar10.load_data()[1][1]
y_train=y_train.reshape(len(y_train))
y_test=y_test.reshape(len(y_test))
num_cluster=[50]
window_size=[2,3,5,7,11]
for size in window_size:
    print(size)
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]

    for K in num_cluster:
        print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        for c in range(10):
            print(c)
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num*3:num*3+3]=train[:,i:32-size+1+i,j:32-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num*3:num*3+3]=X_test[:,i:32-size+1+i,j:32-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
        print(Classic_result)
        print(GMM_result)
        print(IF_result)
        print(LOF_result)
        print(OCSVM_result)

2
50
0
1
2
3
4
5
6
7
8
9
[40.6, 76.4, 40.1, 59.3, 48.5, 58.9, 57.6, 65.2, 67.0, 78.5]
[52.6, 74.7, 49.6, 64.5, 55.3, 66.9, 65.3, 60.9, 71.9, 74.6]
[66.8, 74.8, 64.2, 64.1, 65.9, 67.4, 62.0, 58.2, 73.9, 73.8]
[46.2, 76.5, 54.6, 60.8, 54.9, 59.7, 67.5, 61.3, 65.9, 74.0]
[34.9, 74.9, 36.1, 59.5, 47.4, 57.6, 57.4, 63.0, 61.1, 77.2]
3
50
0
1
2
3
4
5
6
7
8
9
[41.8, 76.7, 39.9, 59.3, 48.4, 60.6, 57.4, 65.2, 65.3, 78.4]
[53.3, 76.9, 48.1, 64.9, 59.1, 68.9, 70.2, 62.4, 70.2, 75.9]
[63.8, 72.8, 64.3, 61.6, 71.1, 63.7, 70.7, 54.9, 72.1, 75.1]
[48.8, 77.5, 53.5, 61.0, 54.0, 62.2, 65.0, 62.3, 62.4, 74.3]
[36.3, 75.7, 35.7, 59.2, 46.2, 59.0, 56.2, 63.6, 59.3, 77.0]
5
50
0
1
2
3
4
5
6
7
8
9
[43.7, 75.3, 40.1, 58.9, 47.8, 62.0, 57.5, 65.4, 62.7, 76.8]
[54.6, 76.6, 50.0, 64.1, 60.1, 66.9, 72.8, 64.9, 69.8, 77.4]
[62.6, 63.0, 64.6, 57.8, 69.5, 57.0, 72.5, 57.4, 73.2, 68.8]
[51.4, 78.5, 53.8, 61.5, 52.4, 65.9, 64.0, 64.0, 62.3, 74.7]
[39.0, 75.8, 36.2, 59.4, 44.1, 60.9, 54.9, 64.9, 57.8, 75.8]
7
50
0
1
2

In [10]:
X_train=cifar10.load_data()[0][0]
y_train=cifar10.load_data()[0][1]
X_test=cifar10.load_data()[1][0]
y_test=cifar10.load_data()[1][1]
y_train=y_train.reshape(len(y_train))
y_test=y_test.reshape(len(y_test))
num_cluster=[25]
window_size=[7]
for size in window_size:
    print(size)
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]

    for K in num_cluster:
        print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        for c in range(10):
            print(c)
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num*3:num*3+3]=train[:,i:32-size+1+i,j:32-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            #makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num*3:num*3+3]=X_test[:,i:32-size+1+i,j:32-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
        print(Classic_result)
        print(GMM_result)
        print(IF_result)
        print(LOF_result)
        print(OCSVM_result)

7
25
0
1
2
3
4
5
6
7
8
9
[45.4, 71.2, 41.8, 57.7, 45.9, 61.5, 53.3, 65.7, 60.6, 73.4]
[56.7, 75.9, 53.3, 61.8, 56.2, 66.5, 68.6, 65.7, 68.7, 72.6]
[60.1, 70.2, 61.7, 60.4, 64.9, 59.5, 71.3, 59.3, 67.7, 69.7]
[55.6, 76.9, 57.6, 61.2, 54.2, 68.5, 63.4, 64.2, 60.4, 71.0]
[43.4, 73.0, 39.5, 57.6, 43.7, 61.9, 52.7, 66.0, 59.5, 72.8]


In [6]:
X_train=fmnist.load_data()[0][0]
y_train=fmnist.load_data()[0][1]
X_test=fmnist.load_data()[1][0]
y_test=fmnist.load_data()[1][1]
num_cluster=[2,3,5,8,10,16]
window_size=[5,7,11,15]
for size in window_size:
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]
    for K in num_cluster:
        print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        #result=[]
        for c in range(10):
            print(c)
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num]=train[:,i:28-size+1+i,j:28-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),28-size+1,28-size+1,size*size])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num]=X_test[:,i:28-size+1+i,j:28-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
    makefile(Classic_table,"result/FMNIST_OCLCB-Classic-size"+str(size)+".pkl")
    makefile(GMM_table,"result/FMNIST_OCLCB-GMM-size"+str(size)+".pkl")
    makefile(IF_table,"result/FMNIST_OCLCB-IF-size"+str(size)+".pkl")
    makefile(LOF_table,"result/FMNIST_OCLCB-LOF-size"+str(size)+".pkl")
    makefile(OCSVM_table,"result/FMNIST_OCLCB-OCSVM-size"+str(size)+".pkl")

2
0
1
2
3
4
5
6
7
8
9
3
0
1
2
3
4
5
6
7
8
9
5
0
1
2
3
4
5
6
7
8
9
8
0
1
2
3
4
5
6
7
8
9
10
0
1
2
3
4
5
6
7
8
9
16
0
1
2
3
4
5
6
7
8
9
2
0
1
2
3
4
5
6
7
8
9
3
0
1
2
3
4
5
6
7
8
9
5
0
1
2
3
4
5
6
7
8
9
8
0
1
2
3
4
5
6
7
8
9
10
0
1
2
3
4
5
6
7
8
9
16
0
1
2
3
4
5
6
7
8
9
2
0
1
2
3
4
5
6
7
8
9
3
0
1
2
3
4
5
6
7
8
9
5
0
1
2
3
4
5
6
7
8
9
8
0
1
2
3
4
5
6
7
8
9
10
0
1
2
3
4
5
6
7
8
9
16
0
1
2
3
4
5
6
7
8
9
2
0
1
2
3
4
5
6
7
8
9
3
0
1
2
3
4
5
6
7
8
9
5
0
1
2
3
4
5
6
7
8
9
8
0
1
2
3
4
5
6
7
8
9
10
0
1
2
3
4
5
6
7
8
9
16
0
1
2
3
4
5
6
7
8
9


In [ ]:
X_train=cifar10.load_data()[0][0]
y_train=cifar10.load_data()[0][1]
X_test=cifar10.load_data()[1][0]
y_test=cifar10.load_data()[1][1]
y_train=y_train.reshape(len(y_train))
y_test=y_test.reshape(len(y_test))
num_cluster=[2,3,5,8,10,16]
window_size=[3,5,7,11,17]
for size in window_size:
    print(size)
    Classic_table=[]
    GMM_table=[]
    IF_table=[]
    LOF_table=[]
    OCSVM_table=[]

    for K in num_cluster:
        print(K)
        kmeans=KMeans(n_clusters=K)
        GMM=GaussianMixture(n_components=1)
        IF=IsolationForest()
        LOF=LocalOutlierFactor(novelty=True)
        OCSVM=OneClassSVM()
        Classic_result=[]
        GMM_result=[]
        IF_result=[]
        LOF_result=[]
        OCSVM_result=[]
        for c in range(10):
            #print(c)
            train = X_train[y_train==c]
            position=np.array(range(size**2)).reshape(size,size)
            windows=np.zeros([len(train),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows[:,:,:,num*3:num*3+3]=train[:,i:32-size+1+i,j:32-size+1+j]
            vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
            kmeans.fit(vector)
            cluster=kmeans.predict(vector)
            cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
            makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
            LCB=[]
            for raw in cluster:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB.append(balance)
            LCB=np.array(LCB)
            GMM.fit(LCB)
            IF.fit(LCB)
            LOF.fit(LCB)
            OCSVM.fit(LCB)
            LP=LCB.mean(axis=0)
            windows_test=np.zeros([len(X_test),32-size+1,32-size+1,size*size*3])
            for i in range(size):
                for j in range(size):
                    num=position[i][j]
                    windows_test[:,:,:,num*3:num*3+3]=X_test[:,i:32-size+1+i,j:32-size+1+j]
            vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
            cluster_test=kmeans.predict(vector_test)
            cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
            LCB_test=[]
            for raw in cluster_test:
                balance=np.zeros(K)
                for value in raw:
                    balance[value]+=1
                LCB_test.append(balance)
            LCB_test=np.array(LCB_test)
            Unseen_score=(abs(LCB_test-LP)).sum(axis=1)
            GMM_seen=GMM.score_samples(LCB_test)
            IF_seen=IF.score_samples(LCB_test)
            LOF_seen=LOF.score_samples(LCB_test)
            OCSVM_seen=OCSVM.score_samples(LCB_test)
            y_eval=np.zeros(len(y_test))
            y_eval[y_test==c]=1
            Classic_result.append(np.round(roc_auc_score(y_eval,Unseen_score*(-1))*100,1))
            GMM_result.append(np.round(roc_auc_score(y_eval,GMM_seen)*100,1))
            IF_result.append(np.round(roc_auc_score(y_eval,IF_seen)*100,1))
            LOF_result.append(np.round(roc_auc_score(y_eval,LOF_seen)*100,1))
            OCSVM_result.append(np.round(roc_auc_score(y_eval,OCSVM_seen)*100,1))
            #result.append(roc_auc_score(y_eval,Unseen_score*(-1)))
        Classic_table.append(Classic_result)
        GMM_table.append(GMM_result)
        IF_table.append(IF_result)
        LOF_table.append(LOF_result)
        OCSVM_table.append(OCSVM_result)
    makefile(Classic_table,"result/CIFAR10_OCLCB-Classic-size"+str(size)+".pkl")
    makefile(GMM_table,"result/CIFAR10_OCLCB-GMM-size"+str(size)+".pkl")
    makefile(IF_table,"result/CIFAR10_OCLCB-IF-size"+str(size)+".pkl")
    makefile(LOF_table,"result/CIFAR10_OCLCB-LOF-size"+str(size)+".pkl")
    makefile(OCSVM_table,"result/CIFAR10_OCLCB-OCSVM-size"+str(size)+".pkl")

3
2
3
5
8
10
16
5
2
3
5
8
10
16
7
2
3
5
8
10
16
11
2
3
5
8
10
16
17
2


In [10]:
LOF_table

[[47.9, 54.7, 49.5, 51.5, 51.1, 50.4, 50.6, 50.8, 52.4, 55.1],
 [47.1, 62.5, 49.8, 49.6, 50.3, 51.1, 54.3, 53.0, 52.8, 59.6],
 [44.7, 69.1, 53.5, 52.4, 53.5, 55.3, 63.0, 53.4, 50.3, 65.3]]

In [33]:
X_train=cifar10.load_data()[0][0]
y_train=cifar10.load_data()[0][1]
X_test=cifar10.load_data()[1][0]
y_test=cifar10.load_data()[1][1]
y_train=y_train.reshape(len(y_train))
y_test=y_test.reshape(len(y_test))

array([6, 9, 9, ..., 9, 1, 1], dtype=uint8)

In [78]:
X_train=cifar10.load_data()[0][0]
y_train=cifar10.load_data()[0][1]
X_test=cifar10.load_data()[1][0]
y_test=cifar10.load_data()[1][1]
y_train=y_train.reshape(len(y_train))
y_test=y_test.reshape(len(y_test))
num_cluster=[6,9,15,24,30,48]
for K in num_cluster:
    print(K)
    kmeans=KMeans(n_clusters=K)
    LOF=LocalOutlierFactor(novelty=True)
    for c in range(10):
        #print(c)
        train = X_train[y_train==c]
        windows=np.zeros([len(train),31,31,12])
        windows[:,:,:,:3]=train[:,:31,:31]
        windows[:,:,:,3:6]=train[:,:31,1:]
        windows[:,:,:,6:9]=train[:,1:,:31]
        windows[:,:,:,9:12]=train[:,1:,1:]
        vector=windows.reshape(windows.shape[0]*windows.shape[1]*windows.shape[2],windows.shape[3])
        kmeans.fit(vector)
        cluster=kmeans.predict(vector)
        cluster=cluster.reshape(windows.shape[0],windows.shape[1]*windows.shape[2])
        makefile(kmeans,"cluster_model/K"+str(K)+"_class"+str(c)+".pkl")
        LCB=[]
        for raw in cluster:
            balance=np.zeros(K)
            for value in raw:
                balance[value]+=1
            LCB.append(balance)
        LCB=np.array(LCB)
        LOF.fit(LCB)
        windows_test=np.zeros([len(X_test),31,31,12])
        windows_test[:,:,:,:3]=X_test[:,:31,:31]
        windows_test[:,:,:,3:6]=X_test[:,:31,1:]
        windows_test[:,:,:,6:9]=X_test[:,1:,:31]
        windows_test[:,:,:,9:12]=X_test[:,1:,1:]
        vector_test=windows_test.reshape(windows_test.shape[0]*windows_test.shape[1]*windows_test.shape[2],windows_test.shape[3])
        cluster_test=kmeans.predict(vector_test)
        cluster_test=cluster_test.reshape(windows_test.shape[0],windows_test.shape[1]*windows_test.shape[2])
        LCB_test=[]
        for raw in cluster_test:
            balance=np.zeros(K)
            for value in raw:
                balance[value]+=1
            LCB_test.append(balance)
        LCB_test=np.array(LCB_test)
        Seen_score=LOF.score_samples(LCB_test)
        y_eval=np.zeros(len(y_test))
        y_eval[y_test==c]=1
        print(np.round(roc_auc_score(y_eval,Seen_score)*100,1))

6
44.5
70.6
53.9
54.4
55.0
53.6
64.8
52.0
50.1
67.7
9
49.9
75.5
56.4
55.9
54.6
57.8
63.7
59.3
53.0
68.3
15
48.1
73.9
55.8
59.6
54.1
58.9
66.2
59.4
60.7
70.2
24
48.1
76.1
55.8
60.0
55.3
60.4
66.6
59.5
63.3
70.9
30
48.1
77.0
54.6
60.3
55.7
60.6
65.9
60.3
66.0
73.1
48
47.9
76.8
54.4
61.2
55.1
60.0
67.3
60.7
66.0
74.3
